# **Training**
Train a Pitch Detection Model and Player Detection Model using YOLOv8. Models are saved locally in `models/` for use in this project (version with git). 

---

## Preparation

**Imports**

In [1]:
from roboflow import Roboflow

**Roboflow API Key**

Before running this block, make sure your Roboflow API keys are stored in `./env/keys.env` as specifed in the [Getting Started](https://github.com/JohnComonitski/FootballTrackingDataGeneration?tab=readme-ov-file#getting-started) section of this repo.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("./../env/keys.env")

# Repo root: directory containing env/, train/, etc. Used for datasets and models.
_cwd = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_cwd, "env", "keys.env")):
        REPO_ROOT = _cwd
        break
    _cwd = os.path.dirname(_cwd)
else:
    REPO_ROOT = os.path.dirname(os.getcwd())

# Access the variables
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [3]:
# Use repo-relative paths so datasets and models stay inside the project
datasets_dir = os.path.join(REPO_ROOT, "train", "datasets")
os.makedirs(datasets_dir, exist_ok=True)
os.chdir(datasets_dir)

---

## Training a Pitch Detection Model

A pitch detection model will be trained using a dataset from Roboflow. The dataset can be found [here](https://universe.roboflow.com/roboflow-jvuqo/football-field-detection-f07vi). Open it and click ***Fork*** to copy it to your workspace, then use the download code from your fork. 

![fork.png](./../examples/fork.png)

Once forked, you will be prompted with a download button, click the download button.

![download.png](./../examples/download.png)

You will now be prompted to select your model type, for this proejct I used YOLOv8. Additionally, check the ***Show download code*** button and uncheck the ***Also train a model for Lable Assist with Roboflow Train*** option, since we will be training the model ourselves.

![ascode.png](./../examples/ascode.png)

The dataset will be forked into your Roboflow workspace.

![code.png](./../examples/code.png)

Replace the code below with the last 3 lines of Python code from your forked project's download snippet.

In [4]:
project = rf.workspace("roboflow-jvuqo").project("football-field-detection-f07vi")
version = project.version(15)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


Before running the next cell, update the numbers to match the version number indicated by this line of code `project.version(1)` given to you.

In [5]:
!sed -i '' 's|\(train: \).*|\1../train/images|' "./football-field-detection-15/data.yaml"
!sed -i '' 's|\(val: \).*|\1../valid/images|' "./football-field-detection-15/data.yaml"

**Pitch Detection Training**

In [6]:
# Use yolov8n-pose (nano), imgsz=640, batch=2 for CPU - yolov8x at 1280px can take 10+ min/batch on CPU
!yolo task=pose mode=train model=yolov8n-pose.pt data={dataset.location}/data.yaml batch=2 epochs=50 imgsz=640 mosaic=0.0 plots=True project={REPO_ROOT}/models name=pitch_detection_model exist_ok=True

/Users/larry/Bremen/FootballTrackingDataGeneration-main/.venv/lib/python3.12/site-packages/ultralytics/nn/tasks.py:567: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return 

**Pitch detection model saved locally.** Weights in `models/pitch_detection_model/weights/` (best.pt, last.pt). Load with `YOLO("models/pitch_detection_model/weights/best.pt")`.

---

In [7]:
# Models are saved to REPO_ROOT/models/ - commit with git
import os
import shutil
import json
from datetime import datetime

run_dir = os.path.join(REPO_ROOT, "models", "pitch_detection_model")
best_weights = os.path.join(run_dir, "weights", "best.pt")
print(f"Pitch model: {best_weights}" if os.path.exists(best_weights) else f"Pitch model (after training): {best_weights}")

# Promote to global best only if this run's validation mAP50(B) is higher than the current global best
results_path = os.path.join(run_dir, "results.csv")
global_best_dir = os.path.join(REPO_ROOT, "models", "pitch_detection_model_best")
meta_path = os.path.join(global_best_dir, "meta.json")
metric_name = "metrics/mAP50(B)"

if os.path.exists(results_path) and os.path.exists(best_weights):
    import pandas as pd

    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    if metric_name in df.columns:
        current_best = float(df[metric_name].max())
        os.makedirs(global_best_dir, exist_ok=True)

        prev_best = float("-inf")
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    prev_best = float(json.load(f).get("best_score", float("-inf")))
            except Exception:
                prev_best = float("-inf")

        if current_best > prev_best:
            target_weights = os.path.join(global_best_dir, "best.pt")
            shutil.copy2(best_weights, target_weights)
            meta = {
                "metric_name": metric_name,
                "best_score": current_best,
                "source_run": run_dir,
                "updated_at": datetime.now().isoformat(),
            }
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"Global best updated to {current_best:.4f}, copied to {target_weights}")
        else:
            print(f"Global best not updated (current={current_best:.4f}, previous={prev_best:.4f})")
    else:
        print(f"Metric '{metric_name}' not found in results.csv; skipping global-best update.")
else:
    print("results.csv or best.pt missing; skipping global-best update.")

Pitch model: /Users/larry/Bremen/FootballTrackingDataGeneration-main/models/pitch_detection_model/weights/best.pt
Global best not updated (current=0.9950, previous=0.9950)


---

## Training a Player Detection Model

Player detection follows the same process. Use [this](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) dataset (fork it, then paste the download code into the cell below).

In [8]:
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(20)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


Before running the next cell, update the numbers to match the version number indicated by this line of code `project.version(1)` given to you.

In [9]:
!sed -i '' 's|\(train: \).*|\1../train/images|' "./football-players-detection-20/data.yaml"
!sed -i '' 's|\(val: \).*|\1../valid/images|' "./football-players-detection-20/data.yaml"

**Player Detection Training**

In [10]:
!yolo task=detect mode=train model=yolov8x.pt data={dataset.location}/data.yaml batch=2 epochs=30 imgsz=640 plots=True project={REPO_ROOT}/models name=player_detection_model exist_ok=True

/Users/larry/Bremen/FootballTrackingDataGeneration-main/.venv/lib/python3.12/site-packages/ultralytics/nn/tasks.py:567: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return 

**Player detection model saved locally**

In [11]:
# Models saved to REPO_ROOT/models/ - commit with git
import os
import shutil
import json
from datetime import datetime

run_dir = os.path.join(REPO_ROOT, "models", "player_detection_model")
best_weights = os.path.join(run_dir, "weights", "best.pt")
print(f"Player model: {best_weights}" if os.path.exists(best_weights) else f"Player model (after training): {best_weights}")

# Promote to global best only if this run's validation mAP50(B) is higher than the current global best
results_path = os.path.join(run_dir, "results.csv")
global_best_dir = os.path.join(REPO_ROOT, "models", "player_detection_model_best")
meta_path = os.path.join(global_best_dir, "meta.json")
metric_name = "metrics/mAP50(B)"

if os.path.exists(results_path) and os.path.exists(best_weights):
    import pandas as pd

    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    if metric_name in df.columns:
        current_best = float(df[metric_name].max())
        os.makedirs(global_best_dir, exist_ok=True)

        prev_best = float("-inf")
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    prev_best = float(json.load(f).get("best_score", float("-inf")))
            except Exception:
                prev_best = float("-inf")

        if current_best > prev_best:
            target_weights = os.path.join(global_best_dir, "best.pt")
            shutil.copy2(best_weights, target_weights)
            meta = {
                "metric_name": metric_name,
                "best_score": current_best,
                "source_run": run_dir,
                "updated_at": datetime.now().isoformat(),
            }
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"Global best (player) updated to {current_best:.4f}, copied to {target_weights}")
        else:
            print(f"Global best (player) not updated (current={current_best:.4f}, previous={prev_best:.4f})")
    else:
        print(f"Metric '{metric_name}' not found in results.csv; skipping global-best update for player model.")
else:
    print("results.csv or best.pt missing; skipping global-best update for player model.")

Player model: /Users/larry/Bremen/FootballTrackingDataGeneration-main/models/player_detection_model/weights/best.pt
Global best (player) updated to 0.8171, copied to /Users/larry/Bremen/FootballTrackingDataGeneration-main/models/player_detection_model_best/best.pt
